# 03. Risk Score Model and Map Visualization

YOLO ?? ??, U-Net ?? ??, DEM ??? ???? ??? rule-based label? ??? XGBoost ??? ??? Folium ??? ?????.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install xgboost shap folium -q

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

PROJECT_DIR = Path('/content/pm-road-risk-map')
if PROJECT_DIR.exists():
    sys.path.append(str(PROJECT_DIR / 'src'))

from road_risk.risk_scoring import add_rule_based_labels
from road_risk.visualization import create_risk_map

## 1. Load model inputs

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
DAMAGE_CSV = DRIVE_ROOT / 'seoul_images/road_damage_summary.csv'
CRACK_CSV = DRIVE_ROOT / 'real_data/pred_results/crack_metrics.csv'
SLOPE_CSV = DRIVE_ROOT / 'real_data/final_slope.csv'
OUTPUT_DIR = DRIVE_ROOT / 'pm_risk_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

damage_df = pd.read_csv(DAMAGE_CSV)
crack_df = pd.read_csv(CRACK_CSV)
slope_df = pd.read_csv(SLOPE_CSV)

for df in [damage_df, crack_df, slope_df]:
    df['image_key'] = df['image'].astype(str).str.replace(r'\.[^.]+$', '', regex=True)

merged = damage_df.merge(crack_df, on='image_key', suffixes=('', '_crack'))
merged = merged.merge(slope_df, on='image_key', suffixes=('', '_slope'))
merged.shape

## 2. Create rule-based labels

In [ ]:
labeled = add_rule_based_labels(merged)
labeled['final_risk_grade'].value_counts().sort_index()

In [ ]:
labeled.to_csv(OUTPUT_DIR / 'risk_labeled_table.csv', index=False, encoding='utf-8-sig')
labeled.head()

## 3. Train XGBoost risk classifier

In [ ]:
feature_cols = [
    'pothole_count',
    'manhole_count',
    'pothole_area_ratio',
    'max_pothole_area_ratio',
    'crack_area_ratio',
    'crack_length_ratio',
    'component_count',
    'max_crack_area_ratio',
    'slope_degree',
]

X = labeled[feature_cols].fillna(0)
y = labeled['final_risk_grade']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(
    objective='multi:softprob',
    num_class=4,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='mlogloss',
    random_state=42,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

## 4. Predict all Seoul images

In [ ]:
labeled['pred_risk_grade'] = model.predict(X)
proba = model.predict_proba(X)
labeled['pred_confidence'] = proba.max(axis=1)
labeled.to_csv(OUTPUT_DIR / 'risk_predictions.csv', index=False, encoding='utf-8-sig')
labeled[['image_key', 'final_risk_grade', 'pred_risk_grade', 'pred_confidence']].head()

## 5. Create PM risk map

In [ ]:
# slope_df ?? merged table? latitude / longitude ??? ??? ???.
map_df = labeled.rename(columns={'pred_risk_grade': 'final_risk_grade'})
risk_map = create_risk_map(
    map_df,
    output_html=OUTPUT_DIR / 'pm_risk_map.html',
    lat_col='latitude',
    lon_col='longitude',
    grade_col='final_risk_grade',
)
risk_map